In [1]:
%load_ext autoreload
%autoreload 2

# XCodec2 Audio Tokenizer Example

This notebook demonstrates the XCodec2 audio tokenizer that extracts discrete tokens from audio.

**Key Properties:**
- Input sample rate: 16 kHz
- Output sample rate: 16 kHz (or as specified)
- Architecture: Encoder + Decoder with Vector Quantization


## Setup and Imports


In [2]:
import sys
import torch
import numpy as np
from datasets import load_dataset
from IPython.display import Audio, display
import matplotlib.pyplot as plt

# Add our tokenizer module to path
sys.path.append('../src')

# Import our tokenizer registry
from audio_tokenizers import get_tokenizer

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")


/users/lmantel/benchmark-audio-tokenizer/.venv-xcodec2/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PyTorch version: 2.9.0+cpu
CUDA available: False
Using device: cpu


## Load ESB Diagnostic AMI Dataset

In [3]:
# Load the AMI clean split
print("Loading ESB diagnostic dataset - AMI clean split...")
dataset = load_dataset('esb/diagnostic-dataset', 'ami', split='clean')

print(f"Dataset loaded with {len(dataset)} samples")
print(f"Sample keys: {dataset[0].keys()}")


Loading ESB diagnostic dataset - AMI clean split...


Dataset loaded with 770 samples
Sample keys: dict_keys(['audio', 'ortho_transcript', 'norm_transcript', 'id', 'dataset'])


## Find Long Audio Samples


In [4]:
# Calculate duration for each sample and find the longest ones
durations = []
for i, sample in enumerate(dataset):
    audio_array = sample['audio']['array']
    sr = sample['audio']['sampling_rate']
    duration = len(audio_array) / sr
    durations.append((i, duration, sample['norm_transcript']))

# Sort by duration and get top 5 longest
durations.sort(key=lambda x: x[1], reverse=True)
long_samples = durations[:5]

print("Top 5 longest audio samples:")
for idx, (i, dur, transcript) in enumerate(long_samples):
    print(f"{idx+1}. Sample {i}: {dur:.2f} seconds")
    print(f"   Transcript: {transcript[:100]}..." if len(transcript) > 100 else f"   Transcript: {transcript}")
    print()


Top 5 longest audio samples:
1. Sample 85: 11.77 seconds
   Transcript: so that people can go arou go back and forth and choose if or or then again if you just i guess i c ...

2. Sample 1: 11.70 seconds
   Transcript: so i guess we have to reflect on our experiences with remote controls to decide what um we would lik...

3. Sample 100: 11.69 seconds
   Transcript: that f the fruit and vegetable theme is the is the current trend and and therefore um we need to go ...

4. Sample 31: 11.44 seconds
   Transcript: and also we talked about um a location function where maybe you could press a button on the t v and ...

5. Sample 210: 11.33 seconds
   Transcript: the the problem is if you have to go across the building and it adds some overhead every time you wa...



## Initialize XCodec2 Tokenizer


In [5]:
# Initialize the tokenizer
print("Loading XCodec2 tokenizer...")
tokenizer = get_tokenizer('xcodec2', device=device)

print(f"Tokenizer loaded: {tokenizer}")
print(f"  Input sample rate: {tokenizer.sample_rate} Hz")
print(f"  Output sample rate: {tokenizer.output_sample_rate} Hz")
print(f"  Codebook size: {tokenizer.codebook_size:,}")
print(f"  Downsample rate: {tokenizer.downsample_rate}x")


Loading XCodec2 tokenizer...


You are using a model of type xcodec2 to instantiate a model of type xcodec. This is not supported for all configurations of models and can yield errors.


Tokenizer loaded: XCodec2Tokenizer(checkpoint='HKUSTAudio/xcodec2', device='cpu', sample_rate=16000)
  Input sample rate: 16000 Hz
  Output sample rate: 16000 Hz
  Codebook size: 0
  Downsample rate: 0x


## Tokenize and Reconstruct Audio Samples


In [6]:
# Process the top 3 longest samples
results = []

for idx in range(min(3, len(long_samples))):
    sample_idx, duration, transcript = long_samples[idx]
    sample = dataset[sample_idx]
    
    print(f"\n{'='*60}")
    print(f"Processing Sample {idx+1} (index {sample_idx})")
    print(f"Duration: {duration:.2f} seconds")
    print(f"Transcript: {transcript[:150]}..." if len(transcript) > 150 else f"Transcript: {transcript}")
    
    # Get audio data
    audio_array = sample['audio']['array']
    sr = sample['audio']['sampling_rate']
    
    # Convert to tensor
    audio_tensor = torch.from_numpy(audio_array).float()
    
    # Tokenize
    print("\nTokenizing...")
    tokens, encode_info = tokenizer.encode(audio_tensor, sr=sr)
    
    # Show token information
    print(f"Token shape: {tokens.shape}")
    print(f"Number of tokens: {tokens.numel()}")
    print(f"Tokens per second: {tokens.numel() / duration:.1f}")
    
    # Show first 20 discrete token values
    token_values = tokens.squeeze().cpu().numpy()
    if token_values.ndim > 1:
        # Handle multi-dimensional token arrays
        token_values = token_values.flatten()
    print(f"\nFirst 20 discrete token IDs:")
    print(token_values[:20])
    print(f"Token ID range: [{token_values.min()}, {token_values.max()}]")
    print(f"Unique tokens used: {len(np.unique(token_values))}")
    
    # Decode back to audio
    print("\nDecoding tokens back to audio...")
    reconstructed, decode_info = tokenizer.decode(tokens)
    
    print(f"Reconstructed shape: {reconstructed.shape}")
    print(f"Output sample rate: {decode_info.get('output_sample_rate', sr)} Hz")
    
    # Handle shape: (1, 1, T) or (1, T) or (T,)
    rec = reconstructed
    if rec.dim() == 3:
        rec = rec[0, 0]
    elif rec.dim() == 2:
        rec = rec[0]
    rec_np = rec.detach().cpu().numpy()
    
    # Store results
    results.append({
        'original': audio_array,
        'original_sr': sr,
        'reconstructed': rec_np,
        'reconstructed_sr': decode_info.get('output_sample_rate', sr),
        'tokens': token_values,
        'transcript': transcript,
        'duration': duration,
        'encode_time': encode_info.get('encode_time', None),
        'decode_time': decode_info.get('decode_time', None)
    })
    
print(f"\n{'='*60}")
print("Processing complete!")



Processing Sample 1 (index 85)
Duration: 11.77 seconds
Transcript: so that people can go arou go back and forth and choose if or or then again if you just i guess i c i can see like some kind of a thing where like you...

Tokenizing...
Token shape: torch.Size([1, 1, 589])
Number of tokens: 589
Tokens per second: 50.0

First 20 discrete token IDs:
[35967 38933 34895 34951 44295 57625 57689 26160 55901 53279 21065 14896
 60671 23343 37781 61805 45359 45371 59964 53567]
Token ID range: [122, 65504]
Unique tokens used: 572

Decoding tokens back to audio...


OpenBLAS warning: precompiled NUM_THREADS exceeded, adding auxiliary array for thread metadata.
OpenBLAS warning: precompiled NUM_THREADS exceeded, adding auxiliary array for thread metadata.
OpenBLAS warning: precompiled NUM_THREADS exceeded, adding auxiliary array for thread metadata.
To avoid this warning, please rebuild your copy of OpenBLAS with a larger NUM_THREADS setting
OpenBLAS warning: precompiled NUM_THREADS exceeded, adding auxiliary array for thread metadata.
OpenBLAS warning: precompiled NUM_THREADS exceeded, adding auxiliary array for thread metadata.
OpenBLAS warning: precompiled NUM_THREADS exceeded, adding auxiliary array for thread metadata.
OpenBLAS warning: precompiled NUM_THREADS exceeded, adding auxiliary array for thread metadata.
OpenBLAS warning: precompiled NUM_THREADS exceeded, adding auxiliary array for thread metadata.
OpenBLAS warning: precompiled NUM_THREADS exceeded, adding auxiliary array for thread metadata.
OpenBLAS warning: precompiled NUM_THREADS 

BLAS : Bad memory unallocation! :  768  0x400942000000
BLAS : Bad memory unallocation! :  768  0x40096c000000
BLAS : Bad memory unallocation! :  768  0x40096e000000
BLAS : Bad memory unallocation! :  768  0x400970000000
BLAS : Bad memory unallocation! :  768  0x400976000000
BLAS : Bad memory unallocation! :  768  0x40097e000000
BLAS : Bad memory unallocation! :  768  0x400988000000
BLAS : Bad memory unallocation! :  768  0x40098a000000
BLAS : Bad memory unallocation! :  768  0x400996000000
BLAS : Bad memory unallocation! :  768  0x40099a000000
BLAS : Bad memory unallocation! :  768  0x4009a0000000
BLAS : Bad memory unallocation! :  768  0x400980000000
BLAS : Bad memory unallocation! :  768  0x4009a4000000
BLAS : Bad memory unallocation! :  768  0x400972000000
BLAS : Bad memory unallocation! :  768  0x400974000000
BLAS : Bad memory unallocation! :  768  0x40098c000000
BLAS : Bad memory unallocation! :  768  0x400978000000
BLAS : Bad memory unallocation! :  768  0x400982000000
BLAS : Bad

## Play Original vs Reconstructed Audio


In [7]:
# Display audio players for comparison
for idx, result in enumerate(results):
    print(f"\n{'='*60}")
    print(f"Sample {idx+1} - Duration: {result['duration']:.2f}s")
    print(f"Transcript: {result['transcript'][:200]}..." if len(result['transcript']) > 200 else f"Transcript: {result['transcript']}")
    print(f"\nTokens used: {len(result['tokens'])} ({len(result['tokens'])/result['duration']:.1f} tokens/sec)")
    
    if result['encode_time']:
        print(f"Encode time: {result['encode_time']:.3f}s")
    if result['decode_time']:
        print(f"Decode time: {result['decode_time']:.3f}s")
    
    print(f"\nOriginal Audio ({result['original_sr']} Hz):")
    display(Audio(result['original'], rate=result['original_sr']))
    
    print(f"\nReconstructed Audio ({result['reconstructed_sr']} Hz):")
    display(Audio(result['reconstructed'], rate=result['reconstructed_sr']))
    
    # Compression ratio
    original_size = len(result['original']) * 2  # 16-bit audio
    token_size = len(result['tokens']) * 2  # Assume 16-bit tokens
    compression_ratio = original_size / token_size
    print(f"\nCompression ratio: {compression_ratio:.1f}x")



Sample 1 - Duration: 11.77s
Transcript: so that people can go arou go back and forth and choose if or or then again if you just i guess i c i can see like some kind of a thing where like you sort of have like the number come up on the t v l...

Tokens used: 589 (50.0 tokens/sec)
Encode time: 4.976s
Decode time: 0.961s

Original Audio (16000 Hz):



Reconstructed Audio (16000 Hz):



Compression ratio: 319.7x

Sample 2 - Duration: 11.70s
Transcript: so i guess we have to reflect on our experiences with remote controls to decide what um we would like to see in a convenient practical

Tokens used: 586 (50.1 tokens/sec)
Encode time: 4.854s
Decode time: 0.512s

Original Audio (16000 Hz):



Reconstructed Audio (16000 Hz):



Compression ratio: 319.5x

Sample 3 - Duration: 11.69s
Transcript: that f the fruit and vegetable theme is the is the current trend and and therefore um we need to go for that if we want

Tokens used: 585 (50.0 tokens/sec)
Encode time: 4.991s
Decode time: 0.501s

Original Audio (16000 Hz):



Reconstructed Audio (16000 Hz):



Compression ratio: 319.7x


## Summary Statistics


In [8]:
# Calculate summary statistics
print("Summary of Tokenization Results:")
print("="*60)

total_duration = sum(r['duration'] for r in results)
total_tokens = sum(len(r['tokens']) for r in results)

print(f"Total audio processed: {total_duration:.2f} seconds")
print(f"Total tokens generated: {total_tokens:,}")
print(f"Average tokens per second: {total_tokens/total_duration:.1f}")

if results[0]['encode_time']:
    avg_encode_time = np.mean([r['encode_time'] for r in results if r['encode_time']])
    print(f"Average encode time: {avg_encode_time:.3f}s")
if results[0]['decode_time']:
    avg_decode_time = np.mean([r['decode_time'] for r in results if r['decode_time']])
    print(f"Average decode time: {avg_decode_time:.3f}s")

# Calculate average compression ratio
avg_compression = np.mean([
    (len(r['original']) * 2) / (len(r['tokens']) * 2) 
    for r in results
])
print(f"Average compression ratio: {avg_compression:.1f}x")

print(f"\nXCodec2 Properties:")
print(f"  Input sample rate: {tokenizer.sample_rate} Hz")
print(f"  Output sample rate: {tokenizer.output_sample_rate} Hz")
print(f"  Codebook size: {tokenizer.codebook_size:,}")
print(f"  Downsample rate: {tokenizer.downsample_rate}x")
bits_per_token = np.log2(tokenizer.codebook_size)
print(f"  Bits per token: ~{bits_per_token:.1f} (log2({tokenizer.codebook_size}))")
token_rate = total_tokens / total_duration
bitrate = token_rate * bits_per_token / 1000
print(f"  Estimated bitrate: {bitrate:.2f} kbps")


Summary of Tokenization Results:
Total audio processed: 35.16 seconds
Total tokens generated: 1,760
Average tokens per second: 50.1
Average encode time: 4.940s
Average decode time: 0.658s
Average compression ratio: 319.6x

XCodec2 Properties:
  Input sample rate: 16000 Hz
  Output sample rate: 16000 Hz
  Codebook size: 0
  Downsample rate: 0x
  Bits per token: ~-inf (log2(0))
  Estimated bitrate: -inf kbps


/tmp/ipykernel_17375/4082207672.py:31: RuntimeWarning: divide by zero encountered in log2
  bits_per_token = np.log2(tokenizer.codebook_size)


## Summary

XCodec2 tokenizer successfully:
- Encodes audio to discrete tokens
- Uses vector quantization with a codebook
- Can decode tokens back to audio
- Provides efficient compression for audio representation

The discrete tokens can be used for:
- Audio compression and storage
- Input to language models for audio generation
- Audio editing and manipulation tasks
- Building blocks for audio synthesis pipelines
